# ParticleGym quick start

Four short steps: a room evacuation, many rooms at once, an arcade game
through the Gymnasium interface, and a policy trained in a few seconds.
The notebook runs on a CPU in about two minutes.

```bash
pip install -e ".[viz,gymnasium]"
```

In [ ]:
import jax
import jax.numpy as jnp
from IPython.display import Image

from particlegym import envs, viz
from particlegym import physics as pp

## 1. A room evacuation

An environment is an object with two pure functions, `reset` and `step`.
An action sets the direction in which a pedestrian *wants* to walk; the
motion follows from the social force model. Here every pedestrian wants to
walk to the door.

In [ ]:
# A 10 m x 10 m room with a door in the right wall and 30 pedestrians.
env = envs.Evacuation(room=(10.0, 10.0), exits=[pp.Opening("x+", (4.4, 5.6))], n_agents=30)
step = jax.jit(env.step)
directions = jnp.asarray(env.directions)  # the 8 action directions
door = jnp.asarray(env.door_targets)[0]


def to_the_door(state):
    heading = door - state.particles.pos[: env.n_agents]
    return jnp.argmax(heading @ directions.T, axis=1)


key = jax.random.key(0)
obs, state = env.reset(key)
frames = [env.render(state, 240)]
for i in range(150):  # 15 s
    obs, state, reward, terminated, truncated, info = step(
        jax.random.fold_in(key, i), state, to_the_door(state)
    )
    if i % 3 == 2:
        frames.append(env.render(state, 240))
viz.save_gif("room.gif", frames)
print(f"{int(terminated.sum())} of {env.n_agents} pedestrians have left the room")
Image("room.gif")

## 2. Many rooms at once

`reset` and `step` compile with `jax.jit` and batch with `jax.vmap`, so a
thousand rooms are one array program. On a GPU this is how the throughput
for reinforcement learning is reached.

In [ ]:
keys = jax.random.split(key, 1024)
obs, states = jax.vmap(env.reset)(keys)
actions = jax.random.randint(key, (1024, env.n_agents), 0, 8)
obs, states, reward, *_ = jax.jit(jax.vmap(env.step))(keys, states, actions)
print(obs.shape, reward.shape)  # (1024, 30, 4) (1024, 30)

## 3. A game through the Gymnasium interface

The brick-wall game is built from the same physics: the ball is a particle
that bounces off the walls, the bricks and the paddle. The wrappers give the
environments the interface of Gymnasium; `env_state` is the state of the
functional environment, which the scripted player reads.

In [ ]:
import gymnasium as gym

from particlegym import wrappers

wrappers.register_envs()  # the ids "particlegym/..."; importing the module does this too
game = gym.make("particlegym/BrickWall-v0", render_mode="rgb_array")
obs, info = game.reset(seed=0)
player = game.unwrapped.env  # the functional environment
frames, score = [game.render()], 0.0
for i in range(600):
    action = int(player.scripted_policy(game.unwrapped.env_state)[0])
    obs, reward, terminated, truncated, info = game.step(action)
    score += reward
    if i % 6 == 5:
        frames.append(game.render())
    if terminated or truncated:
        break
viz.save_gif("brick_wall.gif", frames, colors=32)
print(f"{score:.0f} bricks removed in {i + 1} steps")
Image("brick_wall.gif")

## 4. Learning

Swimmers in a field of vortices are rewarded for the altitude they gain.
A swimmer that always steers upwards is trapped by the vortices. The PPO
baseline trains a policy that senses the local vorticity and the heading;
the environment and the learner compile into one program, and five seeds
train at once. This is a short run for illustration; see the documentation
page "Learning" for longer runs and their limits.

In [ ]:
from particlegym.baselines import ppo

swim = envs.Microswimmer(n_agents=16)
params = swim.default_params
config = ppo.PPOConfig(
    n_envs=16, n_steps=64, n_iterations=100, n_minibatches=16, reward_scale=10.0
)
train = jax.jit(jax.vmap(lambda k: ppo.train(k, swim, params, config)))
policies, curves = train(jax.random.split(jax.random.key(1), 5))  # five seeds at once


def altitude(policy_fn):
    stats = ppo.evaluate(jax.random.key(2), swim, params, policy_fn, 16, params.max_steps)
    return stats["episode_return"].mean()


learned = jax.vmap(lambda p: altitude(lambda key, obs, state: ppo.act(p, swim, obs)))(policies)
naive = altitude(lambda key, obs, state: swim.naive_policy())
print("altitude gained in 100 time units")
print("  PPO, five seeds:", [round(float(x), 1) for x in learned])
print(f"  always upwards:  {float(naive):.1f}")

## Where to go next

- `docs/environments.md` and `docs/experimental.md`: all environments.
- `docs/learning.md`: wrappers, baselines and results.
- `examples/gallery`: the scripts behind the animations of the README.
- `docs/physics.md`: the physics core, which can also be used without an
  environment.